# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset Title:** A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa

**Croissant schema URL:** [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata object and print basic information
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Croissant JSON-LD @id: {dataset.metadata.id}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

**In Croissant:**
- Record sets are the main tables or entities in the data package.
- Each record set, field, or column is referenced by its unique `@id`.

**Below:**
- We list all record sets defined in the Croissant schema, showing their `@id` and names.


In [ ]:
# Explore the dataset's record sets.
record_sets_info = dataset.metadata.recordSets

if not record_sets_info:
    print("No record sets found in the metadata. Check if the Croissant schema is correct or loaded.")
else:
    print("Record Sets:")
    for rs in record_sets_info:
        print(f"  @id: {rs.id}, name: {rs.name}")

# For illustration, print out fields for the first record set (if any)
if record_sets_info:
    main_record_set = record_sets_info[0]
    print(f"\nFields in Record Set '@id': {main_record_set.id} ({main_record_set.name}):")
    for f in main_record_set.fields:
        print(f"    Field @id: {f.id} | name: {f.name} | dataType: {f.dataType}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**We'll load all available record sets and display a preview. Each is referenced by its `@id`.**


In [ ]:
# List and store all record set @id values
if record_sets_info:
    record_set_ids = [rs.id for rs in record_sets_info]
else:
    record_set_ids = []

# Load records from each record set into a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print("No records loaded.")

# Choose the first record set as the main example for downstream analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    main_df = dataframes.get(main_record_set_id, pd.DataFrame())
    print(f"\nUsing Record Set @id: {main_record_set_id} for EDA.")
else:
    main_record_set_id = None
    main_df = pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering by criteria, normalizing numeric fields, removing outliers, grouping, etc.
- Use `@id` fields from metadata for references.

**Assumption:** We'll use fields with numeric types such as GAD-7, PHQ-9, or PSQ scores, which are likely found in the survey record set. Adjust as appropriate for your dataset structure.

In [ ]:
# EDA for main record set
# Pick a numeric field and a group-by field, using the metadata's @id
# Here, let's assume GAD-7 score (@id: 'gad7_score') and village name (@id: 'village') exist

numeric_field_id = None
group_field_id = None
if main_df.empty:
    print("Main DataFrame is empty. Check previous steps or dataset structure.")
else:
    # Find potential numeric and grouping fields by scanning column names and metadata
    for f in record_sets_info[0].fields:
        if f.dataType in ['Float', 'Integer']:
            numeric_field_id = f.id
            break
    # Try to find a grouping field (e.g. village or demographic)
    for f in record_sets_info[0].fields:
        if 'village' in f.name.lower():
            group_field_id = f.id
            break

    # Use fallback/default if not found
    if numeric_field_id is None and main_df.columns.size > 0:
        numeric_field_id = main_df.columns[0]
    if group_field_id is None and main_df.columns.size > 1:
        group_field_id = main_df.columns[1]

    print(f"Numeric field chosen for analysis: {numeric_field_id}")
    print(f"Grouping field chosen for analysis: {group_field_id}")

    # Filter records (example: numeric field > threshold)
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (showing first 5):")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group and summarize by group_field_id
    if group_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Visualize the numeric field distribution and group averages
import matplotlib.pyplot as plt
import seaborn as sns

if not main_df.empty:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of Field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Grouped barplot if group_field available
    if group_field_id in main_df.columns:
        grouped_for_plot = main_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_for_plot, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded the metadata and records from the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.
- Overviewed the record sets and fields with their unique `@id` references.
- Extracted tabular data and performed basic exploratory analysis: filtering, normalization, and grouping using field `@id`s.
- Visualized key distributions and group summaries, supporting downstream demographic or psychometric analyses.
- For further research, leverage Croissant's rich metadata structure (`@id`, data types, field relationships) to enable reproducible and FAIR workflows.
